# UCInsure: Climate-Driven Insurance Risk (Notebook Project)

This notebook builds an actuarial-style climate risk pipeline using FEMA NFIP claims data.

It implements:
- Allowed-column ingestion from FEMA OpenFEMA
- Three specialized ML models trained in parallel
- Individual model probability scores
- Quantified Risk: normalized score in [0, 1]
- Model Confidence based on historical records available for the same region

In [ ]:
%pip install --break-system-packages numpy pandas scikit-learn joblib matplotlib

import json
import math
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_URL = "https://www.fema.gov/api/open/v2/FimaNfipClaims"
DATA_CACHE_PATH = Path("data") / "cache" / "FimaNfipClaimsV2.csv"
MODEL_CACHE_DIR = Path("data") / "cache" / "models"

ALLOWED_COLUMNS = (
    "reportedCity",
    "reportedZipCode",
    "latitude",
    "longitude",
    "floodEvent",
    "dateOfLoss",
    "yearOfLoss",
    "floodZoneCurrent",
    "waterDepth",
    "numberOfFloorsInTheInsuredBuilding",
    "occupancyType",
    "primaryResidenceIndicator",
    "buildingPropertyValue",
    "contentsPropertyValue",
    "amountPaidOnBuildingClaim",
    "amountPaidOnContentsClaim",
    "buildingDamageAmount",
)

SAMPLE_ROWS = 250_000
RANDOM_STATE = 42